# 09. Clinical disease dimensions

This notebook summarises the positive clinical disease-dimension analyses from their committed machine-readable outputs: Alzheimer phenotype discovery with external confirmation, clinically supported human TMS symptom circuits translated to mouse prescriptions, and a Parkinson stage hypothesis evaluated in the independent QPN-NC cohort.

The notebook does not reinterpret a generic disease map as a validation result. It reads the exact JSON files produced by the analysis scripts and keeps null definitions, sensitivity results and evidence boundaries visible.

In [ ]:
import json, subprocess, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
LOGS = ROOT / 'outputs' / 'logs'

def read_strict(name):
    def reject(value):
        raise ValueError(f'non-finite JSON constant {value} in {name}')
    return json.loads((LOGS / name).read_text(), parse_constant=reject)

ad = read_strict('reverse_translation_ad_phenotypes.json')
leads = read_strict('reverse_translation_ad_leads_confirmation.json')
tms = read_strict('reverse_translation_symptom_dissociation.json')
pd_stage = read_strict('reverse_translation_pd_stage_progression.json')
qpn = read_strict('reverse_translation_qpn_pd_stage.json')
qpn_surface = read_strict('reverse_translation_qpn_pd_stage_surface.json')
print('Loaded six aggregate result files from', LOGS)

## Optional full rerun

The committed JSON files make every headline number inspectable on a bare clone. Set `RUN_ANALYSES=True` to regenerate the analyses based on public inputs. QPN-NC is restricted and is run separately using the commands and approved input layout in `experiments/reverse_translation/qpn_pd_stage_validation/README.md`.

In [ ]:
RUN_ANALYSES = False
scripts = [
    '09_ad_phenotype_dissociation.py',
    '10_ad_external_leads_confirmation.py',
    '07_symptom_dissociation.py',
    '08_pd_stage_progression.py',
]
if RUN_ANALYSES:
    env = {'PYTHONPATH': str(ROOT / 'src')}
    for script in scripts:
        subprocess.run(
            [sys.executable, str(ROOT / 'experiments/reverse_translation' / script)],
            cwd=ROOT, env={**__import__('os').environ, **env}, check=True,
        )

## 1. Alzheimer phenotypes and frozen LEADS confirmation

The target assignment is typical AD to medial-temporal circuitry, PCA/posterior subtype to visual cortex, and lvPPA/left-temporal subtype to auditory and temporal-association cortex. The discovery analysis uses both tau-PET and VBM. LEADS tests the unchanged assignment in a multisite tau-PET dataset. The confirmation gate requires positive selectivity for all three phenotypes and joint one-sided *p* < 0.05 under both canonical OTTER and the directly relevant regional packs removed.

In [ ]:
ad_specs = [
    ('Discovery tau-PET', ad, 'tau'),
    ('Discovery VBM', ad, 'vbm'),
    ('LEADS subtype vs rest', leads, 'primary_thresholded_subtype_vs_rest'),
    ('LEADS stage-matched', leads, 'unthresholded_stage_matched'),
]
ad_rows = []
for label, source, key in ad_specs:
    row = {'analysis': label}
    for coupling, prefix in [('canonical', 'canonical'), ('no_relevant_anchor_packs', 'packs removed')]:
        result = source['results'][coupling]['parcel_balanced'][key]
        row[f'{prefix} C'] = result['joint_selectivity']
        row[f'{prefix} p'] = result['p_one_sided']
        row[f'{prefix}: all matched selectivities positive'] = all(v > 0 for v in result['row_selectivity'].values())
    ad_rows.append(row)
ad_table = pd.DataFrame(ad_rows).set_index('analysis')
display(ad_table.round(4))
assert leads['confirmation_gate_passed'] is True

## 2. TMS symptom-circuit dissociation

Negative and positive sides of the human atlas define dysphoric and anxiosomatic circuits. The primary mouse statistic uses parcel mass and maximum-statistic correction across the declared 5%, 10%, 20% and 30% tails. The regional-pack control removes the amygdala, cingulate and lateral-prefrontal packs while retaining point anchors and the anchor-warped spatial term.

In [ ]:
h = tms['headline']
tms_table = pd.DataFrame([
    {'configuration': 'canonical OTTER', **h['canonical_primary_top_10']},
    {'configuration': 'relevant packs removed', **h['pack_out_primary_top_10']},
    {'configuration': 'matched no connectivity', **h['no_connectivity_primary_top_10']},
    {'configuration': 'paired canonical - no connectivity', **h['paired_primary_top_10']},
]).set_index('configuration')
display(tms_table.round(4))
print('Structure-balanced paired connectivity increment p =',
      round(tms['paired_canonical_minus_no_connectivity']['structure_balanced']['thresholded']['top_10']['p_one_sided_max_fwer'], 4))
print('Threshold-free target-conditional canonical p =',
      round(tms['couplings']['canonical']['weightings']['parcel_mass']['conditional_axis']['threshold_free_signed']['p_one_sided'], 4))

## 3. Parkinson stage translation and QPN-NC validation

The ENIGMA stage maps motivate a directional interoceptive-versus-primary-motor hypothesis. QPN-NC evaluates the same direction in participant-level, control-normative cortical-thinning maps and at two cortical resolutions. All analyses are cross-sectional rather than within-person measures of progression. The committed QPN-NC outputs contain aggregate statistics only.

In [ ]:
pd_canonical = pd_stage['couplings']['pi_canonical.npy']
pd_table = pd.DataFrame({
    'stage': ['HY1', 'HY2', 'HY3', 'HY4/5'],
    'interoceptive - motor bias': [
        pd_canonical['stage_bias']['HY1'],
        pd_canonical['stage_bias']['HY2'],
        pd_canonical['stage_bias']['HY3'],
        pd_canonical['stage_bias']['HY45'],
    ],
})
display(pd_table.round(3))
print(f"ENIGMA linear trend = {pd_canonical['linear_trend']:.3f}; two-sided spatial p = {pd_canonical['p_two_sided']:.4f}")
qpn_rows = []
for name, result in qpn['stage_formulations'].items():
    qpn_rows.append({
        'formulation': name,
        'effect': result.get('D', result.get('rho')),
        'one-sided spatial p': result['spatial_p_one_sided'],
    })
display(pd.DataFrame(qpn_rows).set_index('formulation').round(4))
print('Synchronized maximum-statistic p =', round(qpn['synchronized_max_statistic']['spatial_p_one_sided'], 4))
resolution = pd.DataFrame([
    {'resolution': '50-region', 'D': qpn['HY4_5_OLS']['D'], **qpn['HY4_5_OLS']['hemisphere_D']},
    {'resolution': '148-region', 'D': qpn_surface['primary_HY_surface']['D'], **qpn_surface['primary_HY_surface']['hemisphere_D']},
]).set_index('resolution')
display(resolution.round(3))

## 4. Compact visual summary

These panels are descriptive views of the committed statistics. Inferential definitions remain in the producing JSON files and scripts.

In [ ]:
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), constrained_layout=True)

x = np.arange(len(ad_table))
axes[0].plot(x, ad_table['canonical C'], 'o-', color='#0072B2', label='canonical')
axes[0].plot(x, ad_table['packs removed C'], 'o--', color='#E69F00', label='packs removed')
axes[0].axhline(0, color='0.75', lw=1)
axes[0].set_xticks(x, ['tau', 'VBM', 'LEADS', 'stage-matched'], rotation=25, ha='right')
axes[0].set_ylabel('joint selectivity, C')
axes[0].set_title('Alzheimer phenotypes')
axes[0].legend(frameon=False, fontsize=8)

labels = ['canonical', 'packs removed', 'no connectivity']
values = [h['canonical_primary_top_10']['contrast_C'], h['pack_out_primary_top_10']['contrast_C'], h['no_connectivity_primary_top_10']['contrast_C']]
axes[1].bar(np.arange(3), values, color=['#0072B2', '#E69F00', '0.7'], width=0.65)
axes[1].axhline(0, color='0.75', lw=1)
axes[1].set_xticks(np.arange(3), labels, rotation=25, ha='right')
axes[1].set_ylabel('dysphoric - anxiosomatic C')
axes[1].set_title('TMS symptom circuits')

axes[2].plot(np.arange(4), pd_table['interoceptive - motor bias'], 'o-', color='#009E73')
axes[2].axhline(0, color='0.75', lw=1)
axes[2].set_xticks(np.arange(4), pd_table['stage'])
axes[2].set_ylabel('interoceptive - motor bias')
axes[2].set_title('Parkinson stage')
plt.show()

## 5. Evidence summary

The analyses provide complementary evidence: Alzheimer phenotype assignments are supported in LEADS, the TMS dissociation persists after removing directly relevant regional packs, and the Parkinson stage direction is supported in the independent QPN-NC cohort.

In [ ]:
summary = pd.DataFrame([
    {'analysis': 'Alzheimer phenotypes', 'tier': 'largely independent external confirmation', 'primary positive': leads['confirmation_gate_passed']},
    {'analysis': 'TMS symptom circuits', 'tier': 'clinically supported human inputs; translated-target inference', 'primary positive': h['canonical_primary_top_10']['p_max_fwer'] < 0.05 and h['pack_out_primary_top_10']['p_max_fwer'] < 0.05},
    {'analysis': 'Parkinson stage', 'tier': 'independent QPN-NC validation', 'primary positive': qpn['stage_formulations']['participant_partial_spearman']['spatial_p_one_sided'] < 0.05 and qpn['stage_formulations']['HY4_5_Huber']['spatial_p_one_sided'] < 0.05 and qpn['synchronized_max_statistic']['spatial_p_one_sided'] < 0.05},
]).set_index('analysis')
display(summary)
assert summary['primary positive'].all()
assert qpn['privacy']['participant_identifiers_written'] is False
assert qpn_surface['privacy']['participant_maps_or_scores_written'] is False
print('ALL COMMITTED RESULT CHECKS PASS')